# Proyecto de Machine Learning

## Predicción de Fallas Vehiculares

### Integrante
- Tu nombre

### Materia
Machine Learning

### Objetivo

Desarrollar un modelo de Machine Learning capaz de predecir si un vehículo presenta una falla utilizando el dataset proporcionado por el docente.

---

## Metodología

Este proyecto seguirá la metodología enseñada en el notebook **ML_Modulo_10_Pipeline_Completo_Seleccion_Modelos.ipynb**.

Etapas del proyecto:

1. Carga del dataset
2. Análisis exploratorio (EDA)
3. Limpieza de datos
4. Preprocesamiento
5. Entrenamiento de modelos
6. Comparación de modelos
7. Evaluación
8. Selección del mejor modelo
9. Guardado del modelo

In [ ]:
# ============================================
# LIBRERÍAS DEL PROYECTO
# ============================================

import os

import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

In [ ]:
# ============================================
# CARGAR EL DATASET
# ============================================

df = pd.read_csv("../data/Dataset_fallas_vehiculares_bolivia (2).csv")

df.head()

In [ ]:
# ============================================
# INFORMACIÓN GENERAL DEL DATASET
# ============================================

print("Número de filas y columnas:")
print(df.shape)

print("\nInformación del dataset:")
df.info()

In [ ]:
# ============================================
# ESTADÍSTICAS DESCRIPTIVAS
# ============================================

df.describe()

In [ ]:
# ============================================
# VALORES FALTANTES
# ============================================

df.isnull().sum()

In [ ]:
# ============================================
# VALORES ÚNICOS DE CADA COLUMNA
# ============================================

for columna in df.columns:
    print(f"\n===== {columna} =====")
    print(df[columna].unique()[:10])

In [ ]:
# ============================================
# REGISTROS DUPLICADOS
# ============================================

duplicados = df.duplicated().sum()

print(f"Registros duplicados: {duplicados}")

In [ ]:
# ============================================
# DISTRIBUCIÓN DE LA VARIABLE OBJETIVO
# ============================================

df["falla_vehiculo"].value_counts()

In [ ]:
# ============================================
# PORCENTAJE DE VALORES FALTANTES
# ============================================

porcentaje_nan = (df.isnull().sum() / len(df)) * 100
porcentaje_nan = porcentaje_nan.sort_values(ascending=False)

print(porcentaje_nan)

In [ ]:
# ============================================
# REGISTROS CON AÑO INVÁLIDO (MAYOR A 2026)
# ============================================

df[df["año_fabricacion"] > 2026]

In [ ]:
# ============================================
# KILOMETRAJES NEGATIVOS
# ============================================

df[df["kilometraje"] < 0]

In [ ]:
# ============================================
# TEMPERATURAS DE MOTOR FUERA DE RANGO
# ============================================

df[df["temperatura_motor_c"] < 0]

In [ ]:
# ============================================
# CORREGIR AÑOS INVÁLIDOS
# ============================================

df.loc[df["año_fabricacion"] > 2026, "año_fabricacion"] = np.nan

print("Años inválidos corregidos.")

In [ ]:
# ============================================
# CORREGIR KILOMETRAJES NEGATIVOS
# ============================================

df.loc[df["kilometraje"] < 0, "kilometraje"] = np.nan

print("Kilometrajes corregidos.")

In [ ]:
# ============================================
# CORREGIR TEMPERATURAS NEGATIVAS
# ============================================

df.loc[df["temperatura_motor_c"] < 0, "temperatura_motor_c"] = np.nan

print("Temperaturas corregidas.")

In [ ]:
# ============================================
# RELLENAR COLUMNAS NUMÉRICAS
# ============================================

# Detectar columnas numéricas compatible con pandas 2 y 3
columnas_numericas = [
    col for col in df.columns if str(df[col].dtype) in ("int64", "float64")
]

for columna in columnas_numericas:
    df[columna] = df[columna].fillna(df[columna].median())

In [ ]:
# ============================================
# RELLENAR COLUMNAS CATEGÓRICAS
# ============================================

# Detectar columnas de texto compatible con pandas 2 y 3
columnas_texto = [
    col for col in df.columns if str(df[col].dtype) in ("object", "str")
]

for columna in columnas_texto:
    df[columna] = df[columna].fillna(df[columna].mode()[0])

In [ ]:
# ============================================
# VERIFICACIÓN FINAL DE VALORES FALTANTES
# ============================================

print(df.isnull().sum())

In [ ]:
# ============================================
# CODIFICAR VARIABLES CATEGÓRICAS
# ============================================

label_encoders = {}

columnas_texto = [
    col for col in df.columns if str(df[col].dtype) in ("object", "str")
]

for columna in columnas_texto:
    le = LabelEncoder()
    df[columna] = le.fit_transform(df[columna])
    label_encoders[columna] = le

print("Variables categóricas codificadas correctamente.")

In [ ]:
# ============================================
# SEPARAR VARIABLES (X e y)
# ============================================

X = df.drop("falla_vehiculo", axis=1)
y = df["falla_vehiculo"]

print("X:", X.shape)
print("y:", y.shape)

In [ ]:
# ============================================
# TRAIN TEST SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

In [ ]:
# ============================================
# MODELO 1: REGRESIÓN LOGÍSTICA
# ============================================

modelo_lr = LogisticRegression(max_iter=1000)
modelo_lr.fit(X_train, y_train)

y_pred_lr = modelo_lr.predict(X_test)

accuracy_lr = accuracy_score(y_test, y_pred_lr)

print("Accuracy:", accuracy_lr)
print("\nMatriz de confusión:\n", confusion_matrix(y_test, y_pred_lr))
print("\nReporte de clasificación:\n", classification_report(y_test, y_pred_lr))

In [ ]:
# ============================================
# MODELO 2: DECISION TREE
# ============================================

modelo_dt = DecisionTreeClassifier(random_state=42)
modelo_dt.fit(X_train, y_train)

y_pred_dt = modelo_dt.predict(X_test)

accuracy_dt = accuracy_score(y_test, y_pred_dt)

print("Accuracy Decision Tree:", accuracy_dt)
print("\nMatriz de confusión:\n", confusion_matrix(y_test, y_pred_dt))
print("\nReporte de clasificación:\n", classification_report(y_test, y_pred_dt))

In [ ]:
# ============================================
# MODELO 3: RANDOM FOREST
# ============================================

modelo_rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
modelo_rf.fit(X_train, y_train)

y_pred_rf = modelo_rf.predict(X_test)

accuracy_rf = accuracy_score(y_test, y_pred_rf)

print("Accuracy Random Forest:", accuracy_rf)
print("\nMatriz de confusión:\n", confusion_matrix(y_test, y_pred_rf))
print("\nReporte de clasificación:\n", classification_report(y_test, y_pred_rf))

In [ ]:
# ============================================
# MODELO 4: SVM
# ============================================

modelo_svm = SVC()
modelo_svm.fit(X_train, y_train)

y_pred_svm = modelo_svm.predict(X_test)

accuracy_svm = accuracy_score(y_test, y_pred_svm)

print("Accuracy SVM:", accuracy_svm)
print("\nMatriz de confusión:\n", confusion_matrix(y_test, y_pred_svm))
print("\nReporte de clasificación:\n", classification_report(y_test, y_pred_svm))

In [63]:
# ============================================
# COMPARACIÓN DE MODELOS
# ============================================

resultados = pd.DataFrame({
    "Modelo": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "SVM"
    ],
    "Accuracy": [
        accuracy_lr,
        accuracy_dt,
        accuracy_rf,
        accuracy_svm
    ]
})

resultados = resultados.sort_values(
    by="Accuracy",
    ascending=False
)

print(resultados)

                Modelo  Accuracy
2        Random Forest    0.6535
0  Logistic Regression    0.6340
3                  SVM    0.6260
1        Decision Tree    0.5615


In [ ]:
# ============================================
# GUARDAR EL MEJOR MODELO Y LOS ENCODERS
#
# Los archivos se guardan en api/modelo/ porque
# la API (FastAPI) los carga desde esa carpeta.
# ============================================

ruta_modelo_dir = "../api/modelo"
os.makedirs(ruta_modelo_dir, exist_ok=True)

# Random Forest fue el modelo con mayor accuracy
joblib.dump(modelo_rf, os.path.join(ruta_modelo_dir, "modelo.pkl"))
joblib.dump(label_encoders, os.path.join(ruta_modelo_dir, "label_encoders.pkl"))

print(f"Modelo y encoders guardados correctamente en: {ruta_modelo_dir}/")